In [ ]:
!nvidia-smi

Sun Aug 30 10:07:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import subprocess, sys

TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
BITSANDBYTES_PIN = "0.49.2"

def pip_install(*specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

pip_install(
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"bitsandbytes=={BITSANDBYTES_PIN}",
)

installing: transformers==4.46.* accelerate==1.1.* bitsandbytes==0.49.2


In [ ]:
import csv, threading, time

GPU_SAMPLES = "/content/gpu_samples.csv"
_sampler = {"thread": None, "stop": None}

def _sample_loop(stop_event, path, interval_s):
    with open(path, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["t", "util_gpu", "mem_used_mib"])
        t0 = time.time()
        while not stop_event.is_set():
            out = subprocess.run(
                ["nvidia-smi",
                 "--query-gpu=utilization.gpu,memory.used",
                 "--format=csv,noheader,nounits"],
                capture_output=True, text=True,
            ).stdout.strip()
            parts = [p.strip() for p in out.split(",")]
            if len(parts) == 2:
                w.writerow([round(time.time() - t0, 2), parts[0], parts[1]])
                fh.flush()
            stop_event.wait(interval_s)

def start_sampler(path=GPU_SAMPLES, interval_s=2):
    if _sampler["thread"] and _sampler["thread"].is_alive():
        print("sampler already running; not starting a second one")
        return
    stop = threading.Event()
    th = threading.Thread(
        target=_sample_loop, args=(stop, path, interval_s), daemon=True,
    )
    th.start()
    _sampler["thread"], _sampler["stop"] = th, stop
    print(f"sampler started -> {path} (every {interval_s}s)")

def stop_sampler():
    if _sampler["stop"]:
        _sampler["stop"].set()
    if _sampler["thread"]:
        _sampler["thread"].join(timeout=5)
    _sampler["thread"], _sampler["stop"] = None, None
    print("sampler stopped")

def read_util_mean(path=GPU_SAMPLES):
    vals = []
    with open(path) as fh:
        for row in csv.DictReader(fh):
            try:
                vals.append(float(row["util_gpu"]))
            except (KeyError, ValueError):
                pass
    return sum(vals) / len(vals) if vals else 0.0

In [ ]:
import time, gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)

def load(dtype: str):
    if dtype == "fp16":
        return AutoModelForCausalLM.from_pretrained(
            MODEL, torch_dtype=torch.float16, device_map="cuda")
    if dtype == "int8":
        qc = BitsAndBytesConfig(load_in_8bit=True)
        return AutoModelForCausalLM.from_pretrained(
            MODEL, quantization_config=qc, device_map="cuda")
    raise ValueError(dtype)

def make_prompt(context_tokens: int) -> str:
    # a filler prompt padded to about context_tokens input tokens
    base = "Summarise the following text in one sentence.\n"
    filler = ("The data center runs many small inference requests all day. " * 400)
    ids = tok(base + filler)["input_ids"][:context_tokens]
    return tok.decode(ids)

def resident_vram_gb() -> float:
    torch.cuda.synchronize()
    return torch.cuda.memory_reserved() / (1024 ** 3)

def profile(model, dtype: str, context: int, new_tokens: int = 128, batch: int = 1):
    prompt = make_prompt(context)
    prompts = [prompt] * batch
    enc = tok(prompts, return_tensors="pt", padding=True).to("cuda")
    # warm-up (compile/allocate), not measured
    _ = model.generate(**enc, max_new_tokens=8, do_sample=False)
    vram = resident_vram_gb()
    start_sampler()
    t0 = time.time()
    out = model.generate(**enc, max_new_tokens=new_tokens, do_sample=False)
    dt = time.time() - t0
    stop_sampler()
    gen_tokens = (out.shape[1] - enc["input_ids"].shape[1]) * batch
    return {
        "dtype": dtype,
        "context": context,
        "vram_gb": round(vram, 3),
        "util_mean": round(read_util_mean(), 1),
        "tokens_per_s": round(gen_tokens / dt, 1),
    }

def free_vram():
    """Hand freed memory back to the driver.

    Delete the model variable yourself first, in the cell, with `del model`.
    Passing it to a helper does not work: the helper deletes its own local name
    while your notebook variable still holds the weights, so nothing is freed
    and empty_cache() has nothing to give back.
    """
    gc.collect()
    torch.cuda.empty_cache()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
rows = []
for dtype in ["fp16", "int8"]:
    model = load(dtype)
    for context in [512, 2048, 4096]:
        row = profile(model, dtype, context)
        print(row)
        rows.append(row)
    del model      # without this the next dtype loads on top of this one
    free_vram()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


sampler started -> /content/gpu_samples.csv (every 2s)
sampler stopped
{'dtype': 'fp16', 'context': 512, 'vram_gb': 3.113, 'util_mean': 54.3, 'tokens_per_s': 29.8}


...

sampler started -> /content/gpu_samples.csv (every 2s)
sampler stopped
{'dtype': 'int8', 'context': 512, 'vram_gb': 1.805, 'util_mean': 24.2, 'tokens_per_s': 5.6}


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


sampler started -> /content/gpu_samples.csv (every 2s)
sampler stopped
{'dtype': 'int8', 'context': 2048, 'vram_gb': 2.035, 'util_mean': 26.5, 'tokens_per_s': 5.5}


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


sampler started -> /content/gpu_samples.csv (every 2s)
sampler stopped
{'dtype': 'int8', 'context': 4096, 'vram_gb': 2.309, 'util_mean': 30.3, 'tokens_per_s': 5.2}


In [ ]:
model = load("fp16")
b1 = profile(model, "fp16", 512, new_tokens=128, batch=1)
b8 = profile(model, "fp16", 512, new_tokens=128, batch=8)
del model
free_vram()
print("batch 1:", b1)
print("batch 8:", b8)
print("tokens/s ratio:", round(b8["tokens_per_s"] / b1["tokens_per_s"], 2))
print("util delta:", round(b8["util_mean"] - b1["util_mean"], 1))

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


sampler started -> /content/gpu_samples.csv (every 2s)
sampler stopped


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


sampler started -> /content/gpu_samples.csv (every 2s)
sampler stopped
batch 1: {'dtype': 'fp16', 'context': 512, 'vram_gb': 3.113, 'util_mean': 66.0, 'tokens_per_s': 30.7}
batch 8: {'dtype': 'fp16', 'context': 512, 'vram_gb': 3.527, 'util_mean': 66.3, 'tokens_per_s': 181.8}
tokens/s ratio: 5.92
util delta: 0.3


In [ ]:
import json
with open("batch_check.json", "w") as f:
    json.dump({"batch1_tokens_per_s": b1["tokens_per_s"],
               "batch8_tokens_per_s": b8["tokens_per_s"]}, f, indent=2)

In [ ]:
import json
with open("profile.json", "w") as f:
    json.dump(rows, f, indent=2)
print("wrote", len(rows), "rows to profile.json")

wrote 6 rows to profile.json


In [ ]:
!wget -O "verify_cell.py" "https://raw.githubusercontent.com/shmoa9/lab-aidc/refs/heads/main/W3D1/verify_cell%20.py"

--2026-08-30 11:06:50--  https://raw.githubusercontent.com/shmoa9/lab-aidc/refs/heads/main/W3D1/verify_cell%20.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4296 (4.2K) [text/plain]
Saving to: ‘verify_cell.py’

verify_cell.py      100%[===================>]   4.20K  --.-KB/s    in 0s      

2026-08-30 11:06:50 (53.4 MB/s) - ‘verify_cell.py’ saved [4296/4296]



In [ ]:
!python verify_cell.py

rows: 6, dtypes: ['fp16', 'int8'], contexts: [512, 2048, 4096]
batch-1 tokens/s: 30.7, batch-8 tokens/s: 181.8
GREEN CHECK: PASS
